In [ ]:
import requests
from nsepython import nse_optionchain_scrapper
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

# Monkey patch to ignore SSL
original_get = requests.get
def unsafe_get(*args, **kwargs):
    kwargs['verify'] = False
    return original_get(*args, **kwargs)
requests.get = unsafe_get

# Fetch Option Chain Data
symbol = "NIFTY"
data = nse_optionchain_scrapper(symbol)

# Spot Price
spot = data['records']['underlyingValue']
expiry_list = data['records']['expiryDates']
print(f"Spot: {spot}")
print("Expiries:", expiry_list)

# UI Widgets
expiry_dropdown = widgets.Dropdown(options=expiry_list, description='Expiry:')
dte_slider = widgets.IntSlider(min=0, max=30, step=1, value=0, description='DTE')

# Option Helper
def get_price(strike, opt_type, expiry_selected):
    for d in data['records']['data']:
        if d['strikePrice'] == strike and d['expiryDate'] == expiry_selected:
            if opt_type in d and d[opt_type]:
                return d[opt_type]['lastPrice']
    return 0

# Plot function
def plot_payoff(expiry_selected, dte):
    clear_output(wait=True)
    display(expiry_dropdown, dte_slider)

    # ATM & structure
    atm = round(spot / 50) * 50
    width = 100
    strikes = {
        "PE_BUY": atm - 2 * width,
        "PE_SELL": atm - width,
        "CE_SELL": atm + width,
        "CE_BUY": atm + 2 * width
    }

    # Prices
    pb = get_price(strikes["PE_BUY"], 'PE', expiry_selected)
    ps = get_price(strikes["PE_SELL"], 'PE', expiry_selected)
    cs = get_price(strikes["CE_SELL"], 'CE', expiry_selected)
    cb = get_price(strikes["CE_BUY"], 'CE', expiry_selected)

    net_credit = (ps - pb) + (cs - cb)
    max_loss = width - net_credit
    lotsize = 50

    lower_be = strikes["PE_SELL"] - net_credit
    upper_be = strikes["CE_SELL"] + net_credit

    print(f"\nNet Credit: ₹{net_credit * lotsize:.2f} | DTE: {dte} days")
    print(f"Max Profit: ₹{net_credit * lotsize:.2f}")
    print(f"Max Loss: ₹{-max_loss * lotsize:.2f}")
    print(f"Breakeven Range: {lower_be:.0f} to {upper_be:.0f}")

    # PnL at expiry
    x = np.arange(atm - 400, atm + 400 + 1, 10)
    payoff = []

    for s in x:
        pe_buy = max(strikes["PE_BUY"] - s, 0) - pb
        pe_sell = max(strikes["PE_SELL"] - s, 0) - ps
        ce_sell = max(s - strikes["CE_SELL"], 0) - cs
        ce_buy = max(s - strikes["CE_BUY"], 0) - cb
        total = pe_buy - pe_sell - ce_sell + ce_buy
        payoff.append(total * lotsize)

    # Plot
    plt.figure(figsize=(10, 6))
    plt.plot(x, payoff, label="Payoff", color='blue')
    plt.axhline(0, color='black', linestyle='--')
    plt.axvline(atm, color='gray', linestyle='--', label=f"ATM: {atm}")
    plt.axvline(lower_be, color='orange', linestyle='--', label=f"Lower BE: {lower_be:.0f}")
    plt.axvline(upper_be, color='orange', linestyle='--', label=f"Upper BE: {upper_be:.0f}")
    plt.axvspan(strikes["PE_SELL"], strikes["CE_SELL"], color='green', alpha=0.2, label='Max Profit Zone')
    plt.axvspan(x[0], strikes["PE_BUY"], color='red', alpha=0.1, label='Max Loss Zone')
    plt.axvspan(strikes["CE_BUY"], x[-1], color='red', alpha=0.1)

    plt.title(f"Iron Condor Payoff - Expiry: {expiry_selected} | DTE: {dte}")
    plt.xlabel("Spot Price")
    plt.ylabel("P&L (₹)")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

# Call on change
expiry_dropdown.observe(lambda change: plot_payoff(expiry_dropdown.value, dte_slider.value), names='value')
dte_slider.observe(lambda change: plot_payoff(expiry_dropdown.value, dte_slider.value), names='value')

# Initial display
display(expiry_dropdown, dte_slider)
plot_payoff(expiry_dropdown.value, dte_slider.value)